In [13]:
# pip install --upgrade google-cloud-bigquery
# pip install pydata-google-auth
# pip install db-dtypes

In [14]:
# Turn off warnings
import warnings
warnings.filterwarnings('ignore')

In [1]:
# Authenticate through your browser with you Big Query account
import pydata_google_auth

scopes = ['https://www.googleapis.com/auth/bigquery']

credentials = pydata_google_auth.get_user_credentials(
    scopes,
    # Set auth_local_webserver to True for a more convenient flow,
    # but be aware it may not work on a remote server
    # like a cloud-hosted notebook if there are network restrictions
    # auth_local_webserver=True, # Optional
)

In [2]:
from google.cloud import bigquery

project = 'product-project-463105'

client = bigquery.Client(project=project,credentials=credentials)
google_citibike = '`bigquery-public-data.new_york_citibike.citibike_trips`'

# Question 2, Part A

In [15]:

# What was the busiest start station by trip count, and how many trips started there?
query = f'''
-- Count distinct trips and use qualify with rank to get the start_station_id with the most trips
SELECT start_station_id, start_station_name, COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) d_trips
FROM
`bigquery-public-data.new_york_citibike.citibike_trips`

WHERE EXTRACT(YEAR FROM starttime) = 2017
GROUP BY 1,2
--ORDER BY 3 DESC

QUALIFY RANK() OVER(ORDER BY COUNT(*) DESC) = 1;

'''

busy_station_df = client.query(query).to_dataframe()
busy_station_df

,start_station_id,start_station_name,d_trips
0,519,Pershing Square North,135906


### Q: What was the busiest start station by trip count, and how many trips started there?
### A: The busiest station is Pershing Square North, adjacent to Grand Central. This makes sense as a commuter hub.<br>

In [20]:

# What was the median trip duration for Subscriber vs Customer? How do you read the gap?
query = f'''
-- median trip length by usertype
SELECT DISTINCT usertype, PERCENTILE_CONT(tripduration, 0.5) OVER(PARTITION BY usertype) AS median_tripduration
FROM
`bigquery-public-data.new_york_citibike.citibike_trips`

WHERE EXTRACT(YEAR FROM starttime) = 2017
AND usertype IS NOT NULL;

'''

median_duration_df = client.query(query).to_dataframe()
# Convert seconds to minutes
median_duration_df['median_minutes'] = (median_duration_df['median_tripduration'] / 60).round(1)
median_duration_df[['usertype','median_minutes']]

,usertype,median_minutes
0,Customer,21.5
1,Subscriber,9.7


### Q: What was the median trip duration for Subscriber vs Customer? How do you read the gap?
### A: Customers ride at a median of ~21.5 minutes vs ~9.7 for Subscribers. The gap reflects different use cases.
### Subscribers are commuters doing short point-to-point trips, Customers are casual/tourist riders on longer leisure trips.<br>

In [31]:

# What share of trips were round trips (same start and end station)? 
query = f'''
-- Get the pct of roundtrip trips using distinct inside the count to make robust to dups
SELECT ROUND(COUNT(DISTINCT CASE WHEN start_station_id = end_station_id THEN bikeid || CAST(starttime AS STRING) END) / 
        COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) * 100.0,1) pct_roundtrip
FROM
`bigquery-public-data.new_york_citibike.citibike_trips` 

WHERE EXTRACT(YEAR FROM starttime) = 2017;

'''

pct_rndtrip_df = client.query(query).to_dataframe()
display(pct_rndtrip_df.style.format({'pct_roundtrip': '{:.0f}%'}))


# Does it differ by user type?
query = f'''
-- Adding usertype cut to the above query
SELECT usertype, ROUND(COUNT(DISTINCT CASE WHEN start_station_id = end_station_id THEN bikeid || CAST(starttime AS STRING) END) / 
        COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) * 100.0,1) pct_roundtrip
FROM
`bigquery-public-data.new_york_citibike.citibike_trips`

WHERE EXTRACT(YEAR FROM starttime) = 2017
GROUP BY 1;

'''

pct_rndtrip_type_df = client.query(query).to_dataframe()
pct_rndtrip_type_df.style.format({'pct_roundtrip': '{:.1f}%'})

,pct_roundtrip
0,2%


,usertype,pct_roundtrip
0,Customer,5.9%
1,Subscriber,1.5%


### Q: What share of trips were round trips (same start and end station)? Does it differ by user type?
### A: 2% overall, 5.9% for Customers vs 1.5% for Subscribers. Citibike is designed for one-way trips. 
### Round trips are almost entirely a leisure behavior, which is why Customers are 4x more likely to do them.

In [37]:
# Among the top 20 start stations by volume, which station has the most lopsided rush-hour split (share of trips starting 7–10am vs 4–7pm on weekdays)? 
# Define “lopsided” however you want, but say what you mean. What might that tell you?
query = f'''
-- q4
WITH t20 AS
(
SELECT start_station_id
FROM
`bigquery-public-data.new_york_citibike.citibike_trips`

WHERE EXTRACT(YEAR FROM starttime) = 2017
GROUP BY 1
QUALIFY RANK() OVER( ORDER BY COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) DESC) <=20
),
am_pm AS
(
SELECT t.start_station_id, t.start_station_name,
  COUNT(DISTINCT
        CASE WHEN CAST(starttime AS TIME) BETWEEN '07:00:00' AND '10:00:00' THEN bikeid || CAST(starttime AS STRING) END
        ) / COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) am_rush_hr,
  COUNT(DISTINCT
        CASE WHEN CAST(starttime AS TIME) BETWEEN '16:00:00' AND '19:00:00' THEN bikeid || CAST(starttime AS STRING) END
        ) / COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) pm_rush_hr
FROM
`bigquery-public-data.new_york_citibike.citibike_trips` t
JOIN 
t20
ON t.start_station_id = t20.start_station_id
WHERE EXTRACT(YEAR FROM starttime) = 2017
AND EXTRACT(DAYOFWEEK FROM starttime) BETWEEN 2 AND 6
GROUP BY 1,2
)
SELECT start_station_id, start_station_name, am_rush_hr, pm_rush_hr, ABS(am_rush_hr - pm_rush_hr) abs_diff

FROM am_pm
--ORDER BY 5 DESC
QUALIFY RANK() OVER(ORDER BY ABS(am_rush_hr - pm_rush_hr) DESC) = 1

'''

am_pm_df = client.query(query).to_dataframe()
am_pm_df.columns = ['start_station_id',	'start_station_name', 'am_rush_hr',	'pm_rush_hr', 'pctg_point_diff']
am_pm_df.style.format({'am_rush_hr': '{:.1%}', 'pm_rush_hr': '{:.1%}','pctg_point_diff': lambda x: f"{x * 100:.1f}"})

,start_station_id,start_station_name,am_rush_hr,pm_rush_hr,pctg_point_diff
0,281,Grand Army Plaza & Central Park S,5.3%,43.7%,38.4


### Q: Among the top 20 start stations by volume, which station has the most lopsided rush-hour split (share of trips starting 7–10am vs 4–7pm on weekdays)?
### Define “lopsided” however you want, but say what you mean. What might that tell you?
### A: Grand Army Plaza is overwhelmingly PM-heavy (43.7% PM vs 5.3% AM). To find lopsidedness, I'm using absolute difference between am and pm rush hours. This way I can easily sort by the magnitude of the difference. 

### The location at Central Park's south entrance near hotels makes it a magnet for after-work and evening leisure activity rather than morning commutes.
### Overall, the heavier rush hours tended to be after work rather than before. This makes sense because people usually try to get to work on a fixed schedule whereas after work, there is more flexibility for leisure activities.
